# 📖 Complete Real-World Data Preprocessing Workflow

This notebook demonstrates a complete, real-world data preprocessing workflow used in machine learning. Our goal is to transform a messy real-world dataset (Google Play Store Apps) into model-ready data.

### **Topics Covered:**
1. Dataset Understanding & Why Mixed Types Matter
2. Exploratory Data Analysis (EDA) & Finding Anomalies
3. Handling Mixed Data Types
4. Duplicate Detection (In-depth)
5. Missing Value Handling (Theory & Practice)
6. Outlier Detection and Treatment
7. Rare Category Handling
8. Feature Engineering & Synthetic Target Creation
9. Encoding Categorical Variables
10. Feature Scaling
11. Multicollinearity Detection (Corr vs. VIF)
12. ColumnTransformer & ML Pipeline
13. Generating Synthetic Datasets for Practice
14. **Beyond the Basics: Feature Selection & Imbalanced Data**
15. Practice Exercises

## 1. Dataset Understanding & The Importance of Data Types

**Why must we handle mixed data types *before* other steps?**
If a numeric column (like `Price` or `Installs`) contains even a single string character (like `$` or `+`), pandas reads the entire column as an `object` (string). If we don't fix this first:
* `df.describe()` won't calculate statistics (mean, min, max) for these columns.
* We cannot mathematically detect outliers.
* We cannot properly visualize distributions or correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('googleplaystore.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\n--- Data Types and Missing Values ---")
df.info()

## 2. Exploratory Data Analysis (EDA) & Finding Anomalies

Before altering the data, let's explore it to find anomalies. We know ratings should be between 1.0 and 5.0. But how would a data scientist discover anomalies if they didn't know the dataset beforehand?

**Inspection Techniques:**
1. **Summary Statistics (`describe()`):** Checking the `min` and `max` values of numerical columns is the easiest way to spot impossible values.
2. **Unique Values (`unique()`):** For columns with a discrete set of expected values, viewing the unique entries immediately highlights weird data.

In [ ]:
# 1. Use describe() to find statistical anomalies
display(df.describe()) 
# Notice the 'max' for Rating is 19.0! This is an immediate red flag.

# 2. Use unique() to see all distinct values
print("\nUnique Rating values:")
print(df['Rating'].unique())

# Find the anomaly where Rating > 5
anomaly = df[df['Rating'] > 5]
display(anomaly)

# Drop the anomalous row
df.drop(anomaly.index, inplace=True)
print(f"Dataset shape after dropping anomaly: {df.shape}")

## 3. Handling Mixed Data Types (Cleaning Strings to Numerics)

Now we convert those "string-like" numeric columns into actual numbers by stripping out text characters.

In [ ]:
# 1. Clean 'Reviews'
df['Reviews'] = df['Reviews'].astype(int)

# 2. Clean 'Installs' (Remove '+' and ',')
df['Installs'] = df['Installs'].str.replace('+', '').str.replace(',', '')
df['Installs'] = pd.to_numeric(df['Installs'])

# 3. Clean 'Price' (Remove '$')
df['Price'] = df['Price'].str.replace('$', '')
df['Price'] = pd.to_numeric(df['Price'])

# 4. Clean 'Size' (Convert 'M' to Megabytes and 'k' to Kilobytes, 'Varies with device' to NaN)
def clean_size(size_val):
    if pd.isna(size_val) or size_val == 'Varies with device':
        return np.nan
    elif 'M' in size_val:
        return float(size_val.replace('M', ''))
    elif 'k' in size_val:
        return float(size_val.replace('k', '')) / 1024.0
    return np.nan

df['Size'] = df['Size'].apply(clean_size)

## 4. Duplicate Detection

A single app can be scraped multiple times. Pandas `drop_duplicates()` is our primary tool here. 
**Understanding the Parameters of `drop_duplicates()`:**
* `subset`: Which columns to consider for identifying duplicates. If None, uses all columns.
* `keep`: 
  * `'first'`: (Default) Keep the first occurrence, drop the rest.
  * `'last'`: Keep the last occurrence.
  * `False`: Drop *all* duplicates entirely.
* `inplace`: If `True`, modifies the dataframe directly instead of returning a copy.
* `ignore_index`: If `True`, resets the row index from 0 to n-1 after dropping.

In [ ]:
# Check total fully duplicated rows
print(f"Total fully duplicated rows: {df.duplicated().sum()}")

# Drop exact duplicates across all columns
df.drop_duplicates(inplace=True)

# Sometimes the same App is recorded twice with varying review counts. 
# We sort by Reviews to keep the entry with the most up-to-date data.
df = df.sort_values('Reviews', ascending=False)

# Drop duplicates based ONLY on the 'App' column, keeping the first (which has max reviews)
df.drop_duplicates(subset=['App'], keep='first', inplace=True, ignore_index=True)

print(f"Dataset shape after handling all duplicates: {df.shape}")

## 5. Missing Value Handling

### The Theory of Missing Data
Before imputing, understand *why* data is missing:
1. **MCAR (Missing Completely At Random):** The missingness has no relationship with any data (e.g., a sensor randomly glitched). Safe to drop or simple impute.
2. **MAR (Missing At Random):** The missingness is related to *other* observed variables (e.g., Men are less likely to fill out a "weight" survey field). Use model-based imputation.
3. **MNAR (Missing Not At Random):** The missingness is related to the missing value itself (e.g., people with very low incomes skip the "income" question). Hardest to solve; requires domain knowledge.

### When to Drop vs. Impute?
* **Drop Columns:** If a column has > 40-50% missing values and lacks high predictive power.
* **Drop Rows:** If the *Target Variable* is missing, or if a row has missing values across almost all columns.
* **Impute (Fill):** * *Mean:* Numeric data with normal distribution (no outliers).
  * *Median:* Numeric data with skewed distribution/outliers (e.g., `Rating`, `Size`).
  * *Mode:* Categorical data.
  * *Advanced:* KNN Imputer or Iterative Imputer (uses ML to predict missing values).

In [ ]:
# Check missing values
print(df.isnull().sum())

# We use Median for Rating because ratings are highly skewed towards 4-5.
df['Rating'].fillna(df['Rating'].median(), inplace=True)

# For 'Size', apps in the same category usually have similar sizes. 
# We use GroupBy imputation (A form of MAR handling).
df['Size'] = df.groupby('Category')['Size'].transform(lambda x: x.fillna(x.median()))

# Drop the remaining few nulls in minor columns (Dropping rows)
df.dropna(subset=['Type', 'Content Rating', 'Current Ver', 'Android Ver', 'Size'], inplace=True)

## 6. Outlier Detection and Treatment

### Detecting Outliers:
1. **Boxplots (IQR Method):** Visualizes the Interquartile Range. Best for standard features.
2. **Z-Score:** Calculates how many standard deviations a point is from the mean. Only works if data is normally distributed.
3. **Isolation Forests:** Machine learning anomaly detection algorithm (great for high-dimensional data).

### Handling Outliers:
1. **Trimming:** Completely dropping extreme rows (Causes data loss).
2. **Capping (Winsorization):** Replacing values beyond a certain threshold (e.g., 99th percentile) with the threshold value.
3. **Transformation:** Using Log, Square Root, or Box-Cox transformations to squash extreme values and normalize the distribution.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['Price'])
plt.title('Price Outliers')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Reviews'])
plt.title('Reviews Outliers')
plt.show()

# 1. Transformation approach for Reviews (Log Transformation for extreme right-skewness)
df['Log_Reviews'] = np.log1p(df['Reviews']) 

# 2. Capping approach for Price (Winsorization at the 99th percentile)
price_99th = df['Price'].quantile(0.99)
df['Price_Capped'] = np.where(df['Price'] > price_99th, price_99th, df['Price'])

## 7. Rare Category Handling

**Why is this important?**
If a categorical column has values that appear only 1 or 2 times, One-Hot Encoding them will create columns consisting almost entirely of 0s. This leads to the **Curse of Dimensionality** (too many features) and causes models to **overfit** to rare instances. We group them into an "Other" bucket.

In [ ]:
# Grouping 'Adults only 18+' and 'Unrated' into an 'Other' bucket
rare_categories = ['Adults only 18+', 'Unrated']
df['Content Rating'] = df['Content Rating'].replace(rare_categories, 'Other/Unrated')

print(df['Content Rating'].value_counts())

## 8. Feature Engineering & Synthetic Target Generation

Creating new features from existing ones can give ML models more predictive power.
Here, we extract datetime features and create a synthetic target column `App_Engagement_Score` to simulate an internal company metric we want to predict.

In [ ]:
# Extract Date Features
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
df['Update_Year'] = df['Last Updated'].dt.year

# Create Synthetic Target (A mix of Ratings, Log Reviews, and Random Noise)
np.random.seed(42)
df['App_Engagement_Score'] = (df['Rating'] * 10) + (df['Log_Reviews'] * 2) + np.random.normal(0, 5, len(df))

# Simple binary feature map
df['Is_Free'] = df['Type'].map({'Free': 1, 'Paid': 0})

## 9. Encoding Categorical Variables
Machine Learning models only understand numbers. We must convert text categories into numerical representations.
* **One-Hot Encoding:** Creates a binary column (0 or 1) for each category. Use for **Nominal** data (no intrinsic order, e.g., Colors, `Category`). 
* **Label / Ordinal Encoding:** Assigns integers (0, 1, 2) to categories. Use for **Ordinal** data (inherent order, e.g., Low, Medium, High).
* **Target Encoding:** Replaces a category with the mean of the target variable for that category. Use for **High Cardinality** nominal features (e.g., Zip Codes).

## 10. Feature Scaling
Models like SVM, KNN, and Neural Networks calculate distances between data points. If `Size` is in the millions and `Rating` is 1-5, the model will ignore `Rating`.
* **StandardScaler:** Transforms data to have a Mean of 0 and Std Dev of 1. Use when data follows a Gaussian (Normal) distribution.
* **MinMaxScaler:** Scales values strictly between 0 and 1. Use for Non-Gaussian data, Neural Networks, and Image processing.
* **RobustScaler:** Uses median and quartiles. Use when your data contains severe outliers that you decided not to remove/cap.

*(Note: We will apply these inside our Scikit-Learn Pipeline below to avoid data leakage).*

## 11. Multicollinearity Detection (Correlation vs. VIF)

Multicollinearity happens when two or more independent features are highly related to each other. This confuses linear models.

**How to detect it?**
1. **Correlation Matrix (Pearson's r):** Fast and easy. Measures linear relationship strictly between **two** variables at a time. If correlation > 0.85, they are highly correlated.
2. **VIF (Variance Inflation Factor):** More advanced. Measures how much a feature can be predicted by *all other features combined*. If VIF > 5 (or 10), drop the feature.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

num_features = ['Size', 'Installs', 'Price_Capped', 'Update_Year', 'Is_Free']
X_num = df[num_features].dropna()

# Correlation Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(X_num.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# VIF Calculation
vif_data = pd.DataFrame()
vif_data["Feature"] = X_num.columns
X_num_with_const = X_num.assign(const=1) 
vif_data["VIF"] = [variance_inflation_factor(X_num_with_const.values, i) 
                   for i in range(X_num_with_const.shape[1])][:-1]

display(vif_data)

## 12. ColumnTransformer and Machine Learning Pipeline

Pipelines prevent data leakage by ensuring that imputation and scaling rules learned from the training data are strictly applied to the test data without looking at it beforehand.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

target = 'App_Engagement_Score'
features = ['Category', 'Content Rating', 'Size', 'Installs', 'Price_Capped', 'Update_Year', 'Is_Free']

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Column Definitions
numeric_cols = ['Size', 'Installs', 'Price_Capped', 'Update_Year', 'Is_Free']
categorical_cols = ['Category', 'Content Rating']

# Preprocessing Steps
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) 
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Full Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train and Evaluate
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")
print(f"Test R2 Score: {r2_score(y_test, predictions):.4f}")

## 13. Generating Synthetic Datasets for Practice

Scikit-learn provides excellent tools to generate synthetic data on demand.
* **`make_regression`**: Creates data continuous targets.
* **`make_classification`**: Creates data for discrete targets.
* **`make_blobs`**: Creates unlabelled clusters of data.

In [ ]:
from sklearn.datasets import make_regression, make_classification, make_blobs

X_reg, y_reg = make_regression(n_samples=500, n_features=5, noise=10.0, random_state=42)
X_clf, y_clf = make_classification(n_samples=500, n_features=10, n_classes=2, random_state=42)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)

plt.figure(figsize=(5, 3))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', edgecolor='k')
plt.title("Synthetic Clustering Data (make_blobs)")
plt.show()

## 14. Beyond the Basics: Important Preprocessing Steps Often Required

While we covered the essentials for this specific dataset, real-world data science often requires a few more crucial steps depending on the problem:

### A. Feature Selection
Sometimes you have hundreds of features, but not all are useful. Keeping useless features increases training time and causes overfitting. 
* **Filter Methods:** Use statistical tests (like Chi-Square for categorical, ANOVA for continuous) to select features *before* modeling.
* **Wrapper Methods:** Evaluate multiple models using different subsets of features (e.g., Recursive Feature Elimination - RFE).
* **Embedded Methods:** Algorithms that perform feature selection inherently during training (e.g., Lasso Regression (L1 Penalty) naturally shrinks useless feature coefficients to 0; Random Forests provide `feature_importances_`).

### B. Handling Imbalanced Datasets
If you are doing a classification task (e.g., Fraud Detection) and 99% of data is 'Not Fraud' and 1% is 'Fraud', an accuracy of 99% means nothing! The model just predicts 'Not Fraud' every time.
* **Undersampling:** Randomly deleting rows from the majority class until it matches the minority class (Causes loss of information).
* **Oversampling (SMOTE):** Synthetic Minority Over-sampling Technique. It uses K-Nearest Neighbors to create fake, but realistic, rows of the minority class.
* **Algorithmic Level:** Use `class_weight='balanced'` in algorithms like Random Forest or Logistic Regression to heavily penalize the model for getting the minority class wrong.

### C. Text Data Preprocessing (NLP)
If you are analyzing the actual text of the user `Reviews` rather than just numerical scores, standard ML preprocessing isn't enough. You would need:
* **Tokenization:** Splitting sentences into individual words.
* **Stop-Word Removal:** Removing common words like "the", "and", "is".
* **Stemming/Lemmatization:** Converting words to their base form (e.g., "running" -> "run").
* **Vectorization:** Converting text to numbers using TF-IDF (Term Frequency-Inverse Document Frequency) or Word Embeddings (Word2Vec).

## 15. Practice Exercises 🏋️‍♂️

**Challenge yourself! Try completing these tasks:**
1. **Explore MNAR:** Assume that 'Price' missing values in a dataset are actually because the apps are extremely expensive and the developers hid the price. How would you handle this computationally?
2. **Robust Scaling:** Swap out the `StandardScaler` in our ML pipeline for `RobustScaler`. Does the R2 score improve? Why might that be?
3. **Imbalanced Classification:** Convert `App_Engagement_Score` into a binary target (`1` if > 60, else `0`). Use `SMOTE` from the `imblearn` library to balance the dataset before training a Logistic Regression model.